In [50]:
!pip install optuna mlflow dagshub seaborn xgboost

In [51]:
import pandas as pd
import numpy as np
import optuna
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import xgboost as xgb
import dagshub
import mlflow
import mlflow.xgboost

In [52]:
dagshub.init(
    repo_owner="Aryanupadhyay23",
    repo_name="Twitter-Sentiment-Analysis-MLOps",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow"
)

mlflow.set_experiment("hyperparameter tuning")

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

<Experiment: artifact_location='mlflow-artifacts:/e8aa851b064b48618f790743afea7af8', creation_time=1771822080177, experiment_id='6', last_update_time=1771822080177, lifecycle_stage='active', name='hyperparameter tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [53]:
df = pd.read_csv("/kaggle/input/datasets/aryanumri/twitter-sentiment/twitter_cleaned (2).csv")

df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print("Classes:", label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [54]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

y_train = np.array(y_train)
y_test = np.array(y_test)

In [55]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

num_classes = len(np.unique(y_train))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Num classes:", num_classes)

Train shape: (41292, 8000)
Test shape: (10323, 8000)
Num classes: 3


In [56]:
def build_xgb(trial):

    return xgb.XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 200, 800),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

In [57]:
def objective(trial):

    model = build_xgb(trial)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        macro = f1_score(y_test, preds, average="macro")
        weighted = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        mlflow.log_params(trial.params)
        mlflow.log_metric("macro_f1", macro)
        mlflow.log_metric("weighted_f1", weighted)
        mlflow.log_metric("accuracy", acc)

    return macro

In [58]:
study = optuna.create_study(
    direction="maximize",
    study_name="xgb_study",
    storage="sqlite:///xgb_optuna.db",
    load_if_exists=True
)

[I 2026-02-23 08:40:24,895] A new study created in RDB with name: xgb_study


In [59]:
with mlflow.start_run(run_name="XGBoost"):

    mlflow.log_param("model_name", "XGBoost")
    mlflow.log_param("vectorizer", "CountVectorizer")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 8000)
    mlflow.log_param("min_df", 2)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("num_features", X_train.shape[1])

    # Hyperparameter tuning
    study.optimize(objective, n_trials=50)

    best_params = study.best_params

    print("Best Macro F1:", study.best_value)
    print("Best Params:", best_params)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_single_split_macro_f1", study.best_value)

    # Train final model
    final_model = xgb.XGBClassifier(
        **best_params,
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X_train, y_train)

    # 5-Fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_macro = []
    cv_acc = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = xgb.XGBClassifier(
            **best_params,
            objective="multi:softprob",
            num_class=num_classes,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        macro = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)

        cv_macro.append(macro)
        cv_acc.append(acc)

        print(f"Fold {fold} - Macro F1: {macro:.4f}")

    mlflow.log_metric("cv_macro_f1_mean", np.mean(cv_macro))
    mlflow.log_metric("cv_macro_f1_std", np.std(cv_macro))
    mlflow.log_metric("cv_accuracy_mean", np.mean(cv_acc))

    # Final test evaluation
    preds_test = final_model.predict(X_test)

    test_macro = f1_score(y_test, preds_test, average="macro")
    test_weighted = f1_score(y_test, preds_test, average="weighted")
    test_accuracy = accuracy_score(y_test, preds_test)

    mlflow.log_metric("test_macro_f1", test_macro)
    mlflow.log_metric("test_weighted_f1", test_weighted)
    mlflow.log_metric("test_accuracy", test_accuracy)

    print("Final Test Macro F1:", test_macro)

    # Artifacts
    report = classification_report(y_test, preds_test, output_dict=True)
    with open("classification_report_xgb.json", "w") as f:
        json.dump(report, f, indent=4)
    mlflow.log_artifact("classification_report_xgb.json")

    cm = confusion_matrix(y_test, preds_test)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix - XGBoost")
    plt.savefig("confusion_matrix_xgb.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_xgb.png")

    study.trials_dataframe().to_csv("optuna_trials_xgb.csv", index=False)
    mlflow.log_artifact("optuna_trials_xgb.csv")

    # Log only best model
    mlflow.xgboost.log_model(final_model, artifact_path="model")

print("XGBoost experiment completed successfully.")

[I 2026-02-23 08:40:36,769] Trial 0 finished with value: 0.7012194514821163 and parameters: {'n_estimators': 782, 'max_depth': 3, 'learning_rate': 0.2216485219322542, 'subsample': 0.8319731833563196, 'colsample_bytree': 0.6221482104274673, 'gamma': 3.392271621231156, 'min_child_weight': 10}. Best is trial 0 with value: 0.7012194514821163.


🏃 View run trial_0 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/66f3c4799782462db8bda0e25022b7be
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 08:40:48,063] Trial 1 finished with value: 0.7153203854908559 and parameters: {'n_estimators': 562, 'max_depth': 4, 'learning_rate': 0.17027381698801713, 'subsample': 0.9203382251600274, 'colsample_bytree': 0.690933783184114, 'gamma': 1.1906393509062823, 'min_child_weight': 4}. Best is trial 1 with value: 0.7153203854908559.


🏃 View run trial_1 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/f13241c760094c1595b21f70e25652f7
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 08:41:03,947] Trial 2 finished with value: 0.7406338339212231 and parameters: {'n_estimators': 476, 'max_depth': 7, 'learning_rate': 0.18169963109589676, 'subsample': 0.6259122235148208, 'colsample_bytree': 0.9917001666219406, 'gamma': 1.7700948379589365, 'min_child_weight': 5}. Best is trial 2 with value: 0.7406338339212231.


...

🏃 View run trial_48 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/8970c80c5bc84ae684d0fc36f41c059a
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6


[I 2026-02-23 08:58:14,477] Trial 49 finished with value: 0.7402128760794504 and parameters: {'n_estimators': 784, 'max_depth': 8, 'learning_rate': 0.1438701789281028, 'subsample': 0.9351778536251119, 'colsample_bytree': 0.6533997485964772, 'gamma': 3.800581167247306, 'min_child_weight': 2}. Best is trial 35 with value: 0.8524963364457635.


🏃 View run trial_49 at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/b156fce972a944b8adb83bb6519cc2e2
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
Best Macro F1: 0.8524963364457635
Best Params: {'n_estimators': 620, 'max_depth': 9, 'learning_rate': 0.24565267408210134, 'subsample': 0.9996106008959242, 'colsample_bytree': 0.9981896988323291, 'gamma': 0.4615298558844305, 'min_child_weight': 1}
Fold 1 - Macro F1: 0.8284
Fold 2 - Macro F1: 0.8281
Fold 3 - Macro F1: 0.8341
Fold 4 - Macro F1: 0.8357
Fold 5 - Macro F1: 0.8311
Final Test Macro F1: 0.8524963364457635


2026/02/23 09:01:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6/runs/d0b52120ea3944a49e18b71f4d87b757
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/6
XGBoost experiment completed successfully.
